# 00: Setup and sanity checks

Prerequisite checks for the run matrix (Cypher + AQL + Gremlin targets, Ollama + Anthropic models, LDBC):

1. Resolve repo root and import the `harness` package.
2. Check Ollama is reachable (the SDK honours `OLLAMA_HOST`) and the matrix's Ollama models are pulled (the only hard requirement for the DB-free notebooks 01-04).
3. Warn (not fail) if the execution databases are down (Postgres, Neo4j, ArangoDB, Gremlin Server) -- they are needed only for notebook 05, and the graph DBs cannot all run at once.
4. Summarise the gold dataset(s) the run matrix will iterate over.

Run this first. If Ollama is down or a model is missing, fix that before `01_translation_run` (whose Anthropic cells need `ANTHROPIC_API_KEY`). For notebook 05 you also need the databases loaded with LDBC SF1.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "eval"))

import os

from harness import CACHE_DIR, METRICS_DIR, RECORDS_DIR, RUN_MATRIX, load_dataset

for d in (RECORDS_DIR, METRICS_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Repo root:', REPO_ROOT)
print('Records:  ', RECORDS_DIR)
print('Run matrix:')
for rc in RUN_MATRIX:
    print(f'  - {rc.dataset} x {rc.target} x {rc.model} ({rc.provider}), validation={rc.validation_mode or "default"}')

Repo root: /Users/ivona.obonova/school/rows2graph/rows2graph
Records:   /Users/ivona.obonova/school/rows2graph/rows2graph/eval/outputs/records
Run matrix:
  - ldbc x cypher x llama3.2:latest (ollama), validation=default
  - ldbc x cypher x qwen3-coder:30b (ollama), validation=default
  - ldbc x cypher x gemma4:26b (ollama), validation=default
  - ldbc x cypher x claude-opus-4-8 (anthropic), validation=default
  - ldbc x aql x llama3.2:latest (ollama), validation=default
  - ldbc x aql x qwen3-coder:30b (ollama), validation=default
  - ldbc x aql x gemma4:26b (ollama), validation=default
  - ldbc x aql x claude-opus-4-8 (anthropic), validation=default
  - ldbc x gremlin x llama3.2:latest (ollama), validation=default
  - ldbc x gremlin x qwen3-coder:30b (ollama), validation=default
  - ldbc x gremlin x gemma4:26b (ollama), validation=default
  - ldbc x gremlin x claude-opus-4-8 (anthropic), validation=default


## Gold dataset summary

What the run matrix will iterate over.

In [2]:
import pandas as pd

datasets = sorted({rc.dataset for rc in RUN_MATRIX})
rows = []
for ds in datasets:
    rows.extend(
        {'dataset': ds, 'id': q.id, 'difficulty': q.difficulty,
         'sql_features': ', '.join(q.sql_features) or '(none)',
         'targets': ', '.join(sorted(q.expected))}
        for q in load_dataset(ds)
    )
summary = pd.DataFrame(rows)
print(f'Total gold queries across matrix datasets: {len(summary)}')
display(summary.pivot_table(index='dataset', columns='difficulty', values='id', aggfunc='count', fill_value=0))
summary

Total gold queries across matrix datasets: 15


difficulty,easy,hard,medium
dataset,,,
ldbc,3,8,4


,dataset,id,difficulty,sql_features,targets
0,ldbc,ldbc_q01,easy,(none),"aql, cypher, gremlin"
1,ldbc,ldbc_q02,easy,temporal,"aql, cypher, gremlin"
2,ldbc,ldbc_q03,easy,(none),"aql, cypher, gremlin"
3,ldbc,ldbc_q04,hard,"aggregation, join, order_limit","aql, cypher, gremlin"
4,ldbc,ldbc_q05,hard,"aggregation, join, order_limit, subquery, union","aql, cypher, gremlin"
5,ldbc,ldbc_q06,medium,join,"aql, cypher, gremlin"
6,ldbc,ldbc_q07,medium,"join, temporal","aql, cypher, gremlin"
7,ldbc,ldbc_q08,hard,"distinct, join","aql, cypher, gremlin"
8,ldbc,ldbc_q09,medium,"join, order_limit","aql, cypher, gremlin"
9,ldbc,ldbc_q10,hard,"aggregation, join, order_limit","aql, cypher, gremlin"


## Ollama liveness

The `ollama` SDK resolves the endpoint from `OLLAMA_HOST` (its own default when unset); the harness passes no host. We confirm every Ollama-backed matrix cell's model is pulled.

In [3]:
import ollama

ollama_cells = [rc for rc in RUN_MATRIX if rc.provider == 'ollama']
if ollama_cells:
    try:
        available = [m.model for m in ollama.Client().list().models]
        print(f'Ollama reachable: {len(available)} model(s) available.')
        for rc in ollama_cells:
            status = 'OK' if rc.model in available else f'MISSING -- run: ollama pull {rc.model}'
            print(f'  {rc.model}: {status}')
    except Exception as exc:
        print(f'WARNING: could not reach Ollama via OLLAMA_HOST (unset -> SDK default): {type(exc).__name__}: {exc}')
        print('Start it with `ollama serve` before running 01_translation_run.')
else:
    print('No Ollama cells in RUN_MATRIX; skipping Ollama check.')

Ollama reachable: 19 model(s) available.
  llama3.2:latest: OK
  qwen3-coder:30b: OK
  gemma4:26b: OK
  llama3.2:latest: OK
  qwen3-coder:30b: OK
  gemma4:26b: OK
  llama3.2:latest: OK
  qwen3-coder:30b: OK
  gemma4:26b: OK


## Execution databases (needed for notebook 05)

Execution accuracy (notebook 05) runs the gold SQL on graphonauts's Postgres and the generated query on the graph DB: Cypher on Neo4j, AQL on ArangoDB (db `graphonauts`), Gremlin on the in-memory TinkerGraph behind Gremlin Server -- all loaded with LDBC SF1. Notebooks 01-04 need no database, and the graph DBs cannot all run at once (memory), so at most one is expected up at a time and warnings here are normal; these checks only **warn**. The ArangoDB check also verifies the mapping-aligned unified edge collections exist (built by `eval/scripts/build_arango_unified_edges.py`); without them every AQL traversal query scores 0.

In [ ]:
from harness import execution as ex

# Credentials/endpoints come from config/servers/*.yaml via harness.execution (env vars still
# override per field) -- the same path notebook 05 and eval/scripts/validate_gold.py use, so the
# checks and the executors agree and a local run needs no exported passwords. Each runner returns
# (rows, runtime, error); we only warn on a non-None error.
def _warn_db(name, err, hint=''):
    if err:
        detail = (err.splitlines() or [''])[0][:90]
        print(f'  {name}: unavailable ({detail}) -- needed only for notebook 05')
    elif hint:
        print(f'  {name}: reachable, but {hint}')
    else:
        print(f'  {name}: reachable')


print('Execution databases (needed for notebook 05; at most one graph DB up at a time):')
_warn_db('Postgres (5433)', ex.run_postgres('SELECT 1')[2])
_warn_db('Neo4j (7687)', ex.run_cypher('RETURN 1 AS ok')[2])

# ArangoDB: confirm auth/connectivity, then that the mapping-aligned unified edge collections
# exist (KNOWS is one) -- without them every AQL traversal query in notebook 05 scores 0.
aql_err = ex.run_aql('RETURN 1')[2]
edges_hint = ''
if not aql_err and ex.run_aql('FOR e IN KNOWS LIMIT 1 RETURN 1')[2]:
    edges_hint = 'unified edges missing -- run eval/scripts/build_arango_unified_edges.py'
_warn_db('ArangoDB (8529, db graphonauts)', aql_err, edges_hint)

_warn_db('Gremlin Server (8182)', ex.run_gremlin('g.V().limit(1)')[2])

ex.close_clients()

Execution databases (needed for notebook 05; at most one graph DB up at a time):
  Postgres (5433): reachable
  Neo4j (7687): reachable
  ArangoDB (8529, db graphonauts): reachable
  Gremlin Server (8182): reachable


Checks done. Continue to `01_translation_run.ipynb`.